In [ ]:
import json
import pandas as pd

with open("../data/jobs_master.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.json_normalize(data)

print(df.shape)
df.head()

In [ ]:
print(df.columns.tolist())
print(df.dtypes)
df.isnull().sum()

In [ ]:
import json
import pandas as pd

with open("../data/jobs_master.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.json_normalize(data)

print(df.shape)
df.head()

In [ ]:
import json
import pandas as pd

with open("../data/jobs_master.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.json_normalize(data)

print(df.shape)
df.head(10)

In [ ]:
print(df.columns.tolist())
print(df.dtypes)
df.isnull().sum()

In [ ]:
import pandas as pd
import re
import html

In [ ]:
# The DataFrame (df) was created in the previous data processing step

print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))

df.head()

In [ ]:
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData types before transformation:")
print(df.dtypes)

print("\nMissing values before transformation:")
print(df.isna().sum())

In [ ]:
rows_before = len(df)

df = df.drop_duplicates(
    subset=["job_title", "company", "location"],
    keep="first"
).copy()

rows_after = len(df)

print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Duplicates removed:", rows_before - rows_after)

In [ ]:
missing_values = [
    "N/A",
    "NA",
    "None",
    "NULL",
    ""
]

df = df.replace(missing_values, pd.NA)

print(df.isna().sum())

In [ ]:
text_columns = [
    "source",
    "job_title",
    "company",
    "city",
    "location",
    "employment_type",
    "experience_level"
]

for col in text_columns:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )

df[text_columns].head()

In [ ]:
def clean_description(value):

    if pd.isna(value):
        return pd.NA

    text = html.unescape(str(value))

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text if text else pd.NA


df["description"] = df["description"].apply(clean_description)

df[["description"]].head()

In [ ]:
df["posted_at"] = pd.to_datetime(
    df["posted_at"],
    format="mixed",
    errors="coerce",
    utc=True
)

print("posted_at type:")
print(df["posted_at"].dtype)

df[["job_title", "posted_at"]].head()

In [ ]:
df["salary"] = pd.to_numeric(
    df["salary"],
    errors="coerce"
).astype("Float64")

print("Salary type:", df["salary"].dtype)
print("Available salaries:", df["salary"].notna().sum())
print("Missing salaries:", df["salary"].isna().sum())

In [ ]:
valid_levels = [
    "Intern",
    "Junior",
    "Mid Level",
    "Senior",
    "Senior Lead",
    "Lead",
    "Manager",
    "Director",
    "Principal"
]

df["experience_level"] = df["experience_level"].where(
    df["experience_level"].isin(valid_levels),
    pd.NA
)

print(df["experience_level"].value_counts(dropna=False))

In [ ]:
remote_text = (
    df["job_title"].fillna("") + " " +
    df["description"].fillna("")
).str.lower()

df["is_remote"] = (
    remote_text
    .str.contains(r"\bremote\b", regex=True)
    .astype("boolean")
)

print(df["is_remote"].value_counts(dropna=False))

df[
    ["job_title", "is_remote"]
].head(10)

In [ ]:
df["has_salary"] = (
    df["salary"]
    .notna()
    .astype("boolean")
)

print(df["has_salary"].value_counts())

In [ ]:
df["posted_date"] = df["posted_at"].dt.date

df["posted_year"] = (
    df["posted_at"]
    .dt.year
    .astype("Int64")
)

df["posted_month"] = (
    df["posted_at"]
    .dt.month
    .astype("Int64")
)

df["posted_day"] = (
    df["posted_at"]
    .dt.day
    .astype("Int64")
)

df[
    [
        "posted_at",
        "posted_date",
        "posted_year",
        "posted_month",
        "posted_day"
    ]
].head()

In [ ]:
print("===== TRANSFORMATION SUMMARY =====")

print("\nFinal number of rows:")
print(len(df))

print("\nFinal data types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum())

print("\nExperience levels:")
print(df["experience_level"].value_counts(dropna=False))

print("\nRemote jobs:")
print(df["is_remote"].value_counts(dropna=False))

print("\nJobs with salary:")
print(df["has_salary"].value_counts(dropna=False))

In [ ]:
preview_columns = [
    "source",
    "job_title",
    "company",
    "city",
    "location",
    "employment_type",
    "salary",
    "experience_level",
    "is_remote",
    "has_salary",
    "posted_at",
    "posted_year",
    "posted_month"
]

df[preview_columns].head(10)

In [ ]:
saudi_cities = [
    "Riyadh",
    "Jeddah",
    "Makkah",
    "Mecca",
    "Madinah",
    "Medina",
    "Dammam",
    "Khobar",
    "Al Khobar",
    "Dhahran",
    "Taif",
    "Tabuk",
    "Abha",
    "Jubail",
    "Yanbu",
    "Najran",
    "Jizan",
    "Buraidah",
    "Hafar Al-Batin"
]


def recover_city(row):

    # Keep existing city
    if pd.notna(row["city"]):
        return row["city"]

    text = " ".join([
        str(row.get("location", "")),
        str(row.get("job_title", "")),
        str(row.get("description", ""))
    ])

    for city in saudi_cities:
        if re.search(
            r"(?<!\w)" + re.escape(city) + r"(?!\w)",
            text,
            re.IGNORECASE
        ):
            return city

    return pd.NA


missing_before = df["city"].isna().sum()

df["city"] = df.apply(recover_city, axis=1)

missing_after = df["city"].isna().sum()

print("Missing cities before:", missing_before)
print("Missing cities after:", missing_after)
print("Cities recovered:", missing_before - missing_after)

print("\nCity distribution:")
print(df["city"].value_counts(dropna=False))

In [ ]:
def clean_company(company):
    if pd.isna(company):
        return pd.NA

    company = re.sub(r"\s+", " ", str(company)).strip()

    # Only fix names that are completely lowercase
    if company.islower():
        company = company.title()

    return company


df["company"] = df["company"].apply(clean_company)

df[["company"]].head(10)

In [ ]:
def extract_experience_level(title, description):

    title = "" if pd.isna(title) else str(title).lower()
    description = "" if pd.isna(description) else str(description).lower()

    # 1. Check job title
    if re.search(r"\bintern(ship)?\b", title):
        return "Intern"

    if re.search(r"\bsenior\s+lead\b", title):
        return "Senior Lead"

    if re.search(r"\bprincipal\b", title):
        return "Principal"

    if re.search(r"\b(?:senior|sr\.?)\b", title):
        return "Senior"

    if re.search(r"\b(?:lead|team lead|leader)\b", title):
        return "Lead"

    if re.search(r"\b(?:manager|manger)\b", title):
        return "Manager"

    if re.search(r"\bdirector\b", title):
        return "Director"

    if re.search(r"\b(?:junior|jr\.?|entry[- ]level)\b", title):
        return "Junior"

    # 2. Check years of experience in description
    match = re.search(
        r"\b(\d+)\s*(?:-|to)\s*(\d+)\s*years?",
        description
    )

    if match:
        years = int(match.group(1))

        if years <= 2:
            return "Junior"
        elif years <= 5:
            return "Mid Level"
        else:
            return "Senior"

    # Example: 5+ years
    match = re.search(
        r"\b(\d+)\s*\+\s*years?",
        description
    )

    if match:
        years = int(match.group(1))

        if years <= 2:
            return "Junior"
        elif years <= 5:
            return "Mid Level"
        else:
            return "Senior"

    # Example: at least 3 years / minimum 5 years
    match = re.search(
        r"\b(?:minimum|at least)\s+(\d+)\s*years?",
        description
    )

    if match:
        years = int(match.group(1))

        if years <= 2:
            return "Junior"
        elif years <= 5:
            return "Mid Level"
        else:
            return "Senior"

    # Example: 3 years of experience
    match = re.search(
        r"\b(\d+)\s*years?\s+(?:of\s+)?experience\b",
        description
    )

    if match:
        years = int(match.group(1))

        if years <= 2:
            return "Junior"
        elif years <= 5:
            return "Mid Level"
        else:
            return "Senior"

    return pd.NA


# Count missing values before
missing_before = df["experience_level"].isna().sum()

# Only transform rows where experience_level is missing
mask = df["experience_level"].isna()

df.loc[mask, "experience_level"] = df.loc[mask].apply(
    lambda row: extract_experience_level(
        row["job_title"],
        row["description"]
    ),
    axis=1
)

# Count missing values after
missing_after = df["experience_level"].isna().sum()

print("Missing before:", missing_before)
print("Missing after:", missing_after)
print("Recovered:", missing_before - missing_after)

print("\nExperience levels:")
print(df["experience_level"].value_counts(dropna=False))

In [ ]:
missing_before = df["experience_level"].isna().sum()

mask = df["experience_level"].isna()

df.loc[mask, "experience_level"] = df.loc[mask].apply(
    lambda row: extract_experience_level(
        row["job_title"],
        row["description"]
    ),
    axis=1
)

# Convert unsuccessful extraction back to missing
df["experience_level"] = df["experience_level"].replace(
    "N/A",
    pd.NA
)

missing_after = df["experience_level"].isna().sum()

print("Missing experience levels before:", missing_before)
print("Missing experience levels after:", missing_after)
print("Experience levels recovered:", missing_before - missing_after)

print("\nExperience level distribution:")
print(df["experience_level"].value_counts(dropna=False))

In [ ]:
def clean_skills(value):
    if pd.isna(value):
        return pd.NA

    skills = [
        skill.strip()
        for skill in str(value).split(",")
        if skill.strip()
    ]

    # Remove duplicate skills while keeping order
    skills = list(dict.fromkeys(skills))

    return ", ".join(skills) if skills else pd.NA


df["final_skills"] = df["final_skills"].apply(clean_skills)

df[["job_title", "final_skills"]].head(10)

In [ ]:
print("===== FINAL DATA QUALITY REPORT =====")

print("\nRows:", len(df))
print("Columns:", len(df.columns))

print("\nMissing values:")
print(
    df.isna()
      .sum()
      .sort_values(ascending=False)
)

print("\nData types:")
print(df.dtypes)

print("\nDuplicate jobs:")
print(
    df.duplicated(
        subset=["job_title", "company", "location"]
    ).sum()
)

In [ ]:
from pathlib import Path

output_dir = Path("../data")
output_dir.mkdir(exist_ok=True)

output_file = output_dir / "jobs_cleaned.json"

df.to_json(
    output_file,
    orient="records",
    force_ascii=False,
    indent=2,
    date_format="iso"
)

print("Data saved successfully!")
print("Rows saved:", len(df))
print("Saved to:", output_file.resolve())

In [ ]:
from pathlib import Path

wrong_file = Path("data/jobs_cleaned.json")

if wrong_file.exists():
    wrong_file.unlink()
    print("Deleted:", wrong_file.resolve())
else:
    print("File not found:", wrong_file.resolve())
    

In [ ]:
from pathlib import Path

wrong_file = Path("jobs_cleaned.json")

if wrong_file.exists():
    wrong_file.unlink()
    print("Deleted:", wrong_file.resolve())
else:
    print("File not found")